# Data Preprocessing

This notebook handles the initial data preprocessing for the ColdFusionX model. We’ll be working with a subset of the AntM2C 10M dataset (~3.7 million records) from `antm2c_10m_part0.csv`, pre-cached as a Pickle file. The goal is to clean and prepare this sample for use across our encoder modules: ID, Text, Image, and Time.

We'll exclude all `deep_features_0` to `deep_features_26` at this stage and optionally include them later for performance benchmarking.

#### Notebook setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Core Libraries
import os
import pandas as pd
import numpy as np

# Visual checks
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
from tqdm import tqdm
import pickle
import json

In [3]:
csv_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
pkl_path = os.path.join(csv_dir, "antm2c_10m_part0_combined.pkl")

# Load Part 0
print("Loading cached data from Part 0...")
df = pd.read_pickle(pkl_path)

# Sanity check
print("Loaded shape:", df.shape)
df.head(5)

Loading cached data from Part 0...
Loaded shape: (3730329, 10)


,user_id,item_id,log_time,label,bill_entity_seq,service_entity_seq,query_entity_seq,item_entity_names,item_title,scene
0,1236562,287924,2023-07-14 00:21:44,0,"椒盐花生,油炸,收款码,移动支付,牛肉炒面,雪糕,金林饭店,青岛,柚子乌龙茶,公交,康师傅,...","营销玩法,红包,游戏玩乐,任务,现金红包,游戏,理财,抽奖,支付宝,转账,集步,现金,赚钱红...","投教基地,信用卡,月月赚,理财服务,流量卡,积分,洗衣液,电话卡,好医保,手机,花呗金,浦发...","饮用水,百岁山,天然矿泉,绿账,消费金,小瓶,付邮购",消费金-百岁山天然矿泉水整箱小瓶饮用水，绿账优淘，天然矿泉水整箱小瓶付邮购,4
1,1181655,425272,2023-07-18 12:45:30,1,"土家酱香饼,官方旗舰,miniso,湛江市,脆皮火腿肠,椰蓉酥,花呗,曼巴特购物广场,江苏,...","营销玩法,红包,任务,现金,集分宝,现金红包,做任务,现金领取,抽奖,集步,签到,打卡,领现...","中国联通,流量卡,手机号码卡,红包,电话卡,营业厅,天天集宝,腾讯,信用卡,社会保障,肯德基...","医疗,好医保,百万医疗",百万医疗免费体验,2
2,663299,604771,2023-08-01 09:28:15,1,"马桶,婚礼吊兰,资料夹,夹子,润眼液,韩国,马桶坐垫,除胶剂,帐单,大嘴猴,水果,白酒概念,...","蚂蚁森林能量,互动小游戏,公益慈善,公益种树,种树,蚂蚁庄园,营销玩法,养小鸡,喂小鸡,小鸡...","手机,境内自驾游旅行险,消费节,荣耀honor,apple,出租,支付宝,消费金,意外险,保...",商品橱窗,逛15秒商品橱窗,2
3,519271,237488,2023-08-03 23:19:06,0,NaN,NaN,NaN,"阿里宝卡,宝卡,流量卡,vip卡",中国联通营业厅电话卡,1
4,1659372,604771,2023-08-01 20:24:21,1,"转账,收款码,乌苏市,余利宝,银行卡,手机扫码支付,花呗,蚂蚁财富,帐单,通达建材商店,芭芭...","营销玩法,红包,互动小游戏,水果,果树,种果树,果农,芭芭农场,蚂蚁庄园,现金红包,养小鸡,...","健康金,里程,手机,福利金,网商银行,二手手机,免押金租赁,回收回收,火车,免费,新疆维吾尔...",商品橱窗,逛15秒商品橱窗,2


In [4]:
# Base path
output_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed"

# Structured paths
config = {
    "processed_df_path": os.path.join(output_dir, "df_processed.pkl"),
    "id_mappings_dir": os.path.join(output_dir, "id_mappings"),
    "image_path_list": os.path.join(output_dir, "image_paths.csv"),
    "scaler_stats": os.path.join(output_dir, "scaler_stats.json"),
    "split_dir": os.path.join(output_dir, "split_features")
}

# Create subfolders if missing
os.makedirs(config["id_mappings_dir"], exist_ok=True)
os.makedirs(config["split_dir"], exist_ok=True)

## Filter and record validation

- Remove rows with missing or invalid core features: `item_id`, `user_id`, `label`
- Drop malformed or null `log_time` entries
- Discard rows with empty or whitespace-only text in key fields like `item_title`
- Optionally remove underrepresented `scene` values

This reduces noise, prevents errors during training, and ensures a stable, well-defined dataset structure.

In [5]:
# Drop rows with nulls in key columns
before = df.shape[0]

df = df.dropna(subset=['item_id', 'user_id', 'label'])

after = df.shape[0]
print(f"Dropped {before - after:,} rows due to missing item_id/user_id/label")

Dropped 0 rows due to missing item_id/user_id/label


In [7]:
# Convert to datetime
df['log_time'] = pd.to_datetime(df['log_time'], errors='coerce')

# Drop rows with invalid timestamps
before = df.shape[0]
df = df.dropna(subset=['log_time'])
after = df.shape[0]

print(f"Dropped {before - after:,} rows due to malformed timestamps")

Dropped 0 rows due to malformed timestamps


In [8]:
# Remove rows where item_title is empty or whitespace
before = df.shape[0]
df['item_title'] = df['item_title'].fillna("").astype(str)
df = df[~df['item_title'].str.strip().eq('')]
after = df.shape[0]

print(f"Dropped {before - after:,} rows due to empty item_title")

Dropped 0 rows due to empty item_title


This confirms that the AntM2C_part0 dataset is clean and ready for preprocessing without requiring any initial repairs or imputations.

## Format and Normalize Columns
- Standardizing and normalizing column formats
- Converting categorical features to the `category` dtype
- Cleaning and preprocessing text fields (e.g., `item_title`, `item_entity_names`)
- Normalizing numerical features (if used later)

In [6]:
# Already done earlier, but reassert type
df['log_time'] = pd.to_datetime(df['log_time'], errors='coerce')

In [7]:
# Convert ID Features to Categorical dtype
id_columns = ['user_id', 'item_id', 'scene', 'bill_entity_seq',
              'query_entity_seq', 'service_entity_seq', 'item_entity_names']

for col in id_columns:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("Converted ID columns to categorical dtype.")

Converted ID columns to categorical dtype.


In [8]:
# Standardise Text Fileds
text_columns = ['item_title', 'item_entity_names']

for col in text_columns:
    df[col] = df[col].astype(str).fillna("unknown").str.strip()

    # Full-width → half-width conversion
    df[col] = df[col].apply(lambda x: ''.join([chr(ord(c) - 65248) if 65281 <= ord(c) <= 65374 else c for c in x]))

    df[col] = df[col].str.lower()

print("Cleaned and normalized text fields.")

Cleaned and normalized text fields.


In [9]:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# df[deep_features] = scaler.fit_transform(df[deep_features])
# Save scaler.mean_, scaler.scale_ to JSON

# Left commented for now, usage depends on whether we decide to add deep features later on.

In [10]:
# Check for constant columns
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
if constant_cols:
    print("Constant columns found:", constant_cols)
else:
    print("No constant columns detected.")

No constant columns detected.


In [12]:
# Preview updated data
df[['user_id', 'item_id', 'scene', 'item_title', 'item_entity_names']].head(3)

,user_id,item_id,scene,item_title,item_entity_names
0,1236562,287924,4,"消费金-百岁山天然矿泉水整箱小瓶饮用水,绿账优淘,天然矿泉水整箱小瓶付邮购","饮用水,百岁山,天然矿泉,绿账,消费金,小瓶,付邮购"
1,1181655,425272,2,百万医疗免费体验,"医疗,好医保,百万医疗"
2,663299,604771,2,逛15秒商品橱窗,商品橱窗


### Column Formatting Summary

The dataset has been successfully cleaned and standardized:

- All ID-based columns (`user_id`, `item_id`, `scene`, etc.) were cast to `category` dtype for efficient encoding.
- Text fields (`item_title`, `item_entity_names`) were normalized:
  - Filled missing values with `"unknown"`
  - Stripped whitespace
  - Converted full-width characters to half-width (for multilingual normalization)
  - Lowercased all text for consistency with case-insensitive tokenizers (e.g., BERT)
- No constant-value columns were found in the dataset.

These formatting steps prepare the data for consistent use across encoder pipelines (ID, Text, Image, Time).

## Map IDs to Integer Indices
- Create index vocabularies for key categorical fields (e.g., `user_id`, `item_id`, `scene`)
- Replace raw values in the dataset with mapped indices
- Save the vocabularies as `.json` files for consistent use in encoding and model inference

In [20]:
def map_ids_to_indices(df, col, save_path):
    # Create mapping: {original_id: index}
    unique_vals = df[col].cat.categories.tolist()
    id2idx = {v: i for i, v in enumerate(unique_vals)}

    # Map values in column to indices
    df[col + "_idx"] = df[col].map(id2idx)

    # Save mapping
    with open(save_path, 'w') as f:
        json.dump(id2idx, f)

    print(f"Mapped '{col}' to indices and saved to {save_path}")
    return df

In [21]:
id_cols_to_encode = ['user_id', 'item_id', 'scene']
for col in id_cols_to_encode:
    save_path = os.path.join(config['id_mappings_dir'], f"{col}_2idx.json")
    df = map_ids_to_indices(df, col, save_path)

Mapped 'user_id' to indices and saved to /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings/user_id_2idx.json
Mapped 'item_id' to indices and saved to /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings/item_id_2idx.json
Mapped 'scene' to indices and saved to /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings/scene_2idx.json


In [24]:
df[['user_id', 'user_id_idx', 'item_id', 'item_id_idx', 'scene', 'scene_idx']].head()

,user_id,user_id_idx,item_id,item_id_idx,scene,scene_idx
0,1236562,41688,287924,13590,4,4
1,1181655,39834,425272,21351,2,2
2,663299,22354,604771,33398,2,2
3,519271,17547,237488,11190,1,1
4,1659372,55873,604771,33398,2,2


Categorical IDs (`user_id`, `item_id`, `scene`) were successfully mapped to integer indices and saved for embedding lookup. These indexed versions are essential for training embedding layers in downstream ID encoders.

## Adding derived time features
- `hour`: Hour of day (0–23)
- `dayofweek`: Day of week (0=Monday, 6=Sunday)
- `is_weekend`: Binary flag (True if Saturday or Sunday)
- `day`: Calendar day (optional, for drift-aware analysis)

Also added **cyclical encodings** (`sin`, `cos`) for the `hour` feature to preserve its circular nature — enabling the model to learn that 23:00 and 00:00 are temporally adjacent.

In [28]:
# Extract temporal components
df['hour'] = df['log_time'].dt.hour
df['dayofweek'] = df['log_time'].dt.dayofweek  # Monday=0, Sunday=6
df['is_weekend'] = df['dayofweek'] >= 5
df['day'] = df['log_time'].dt.date             # for temporal drift plots

print("Extracted hour, dayofweek, is_weekend, and day.")

Extracted hour, dayofweek, is_weekend, and day.


In [29]:
# Adding Cyclical Encoding for Hour
# Normalize to [0, 2π]
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

print("Added cyclical encodings for hour.")

Added cyclical encodings for hour.


In [30]:
df[['log_time', 'hour', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos']].head(3)

,log_time,hour,dayofweek,is_weekend,hour_sin,hour_cos
0,2023-07-14 00:21:44,0,4,False,0.000000e+00,1.000000
1,2023-07-18 12:45:30,12,1,False,1.224647e-16,-1.000000
2,2023-08-01 09:28:15,9,1,False,7.071068e-01,-0.707107


Time features like `hour` are **circular** (i.e., hour 23 and hour 0 are adjacent), but numeric models treat them as far apart unless we encode this structure. Using `sin` and `cos` transforms preserves this circular relationship, improving generalization for models that lack temporal awareness.

In [ ]:
g